In [ ]:
# 1. サンプルデータセットをダウンロード
%cd /content
!git clone https://github.com/wal-afk/drive_sim
%cd drive_sim
!git pull
!git restore .
!git clean -fd
%cd /content/drive_sim

!pip install -U plotly==6.9

In [ ]:
import yaml

from sim.drive_simulator import CarSim

from sim.vehicle import VehicleProp
from sim.mission_base import MissionBase
from sim.goal import GoalLine, GoalCircle
from sim.drawer import SimDrawer, MissionDrawer
from sim.worlds.type_b_world import type_b_circuit
from sim.sign import Sign

with open("config/type-b.yaml", "r") as f:
    vehicle_config = yaml.safe_load(f)

prop = VehicleProp(**vehicle_config)

# ミッション3

下記の命令を組み合わせてプログラムを書き、標識を使ってロボットを誘導することでコースに沿って１周させよう。

## 取り組み方
1. 使える命令を理解する
2. 下のセルを実行して、ロボットの限界速度や、ロボットが存在する初期位置を把握する
3. ２つ下のセル内において
    - プログラムを書く
    - 路面標識を好きな位置に設置する
    - 実行して結果を見る
4. シミュレータでうまく動いたらロボットにプログラムを書きこんで動かしてみよう（講師に声をかけてね）

|使える命令|意味|指定できる値|使い方の例|
|--|--|--|--|
|move|一定速度で前に進む|v=速度[m/秒]|move(v=0.2)|
|rotate|一定速度で回転する|w=回転速度[度/秒]|rotate(w=90)|
|serach|標識を見つける（複数見つかった場合は、最も近いもの）||pos = Search()|
|serach|特定の標識を見つける（複数見つかった場合は、最も近いもの）|name = "標識名"|pos = Search(name="sign1")|

## 注意点
- スタート時の位置はランダムに最大10cmほどずれる
- スタート時の向きはランダムに最大5度ほどずれる

## 路面標識の置き方

- １つ下のセルの`self.set_signs`の部分（下記参照）の中身の`Sign(x=横位置, y=縦位置, name="標識名")`の箇所を自由に書き換える
- 使う標識の名前は下記のいずれかにすること
    - original
        - 今回自分でAIを学習したオリジナルの標識
    - left, right, warn, stop
        - 予め用意されている標識
    - ヒント：　どの標識を使うべきかは、認識精度を踏まえて選ぶとよいかも？ 
      - 標識のデザインとロボットの行動を意味的に一致させる必要はありません。（「stopを見た時に回転する」でもOK）

- 例１：originalの標識を１個（x=1[m],y=0[m]の位置）置く場合
    ```
    self.set_signs(
        [
            Sign(x=1.0, y=0.0, name="original")
        ]
    )
    ```

- 例２：originalの標識を２個（x=1[m],y=0[m]の位置とx=2[m],y=0[m]の位置）、warnの標識を１個（x=3[m], y=1[m]の位置）置く場合
    - 複数並べるには、最後に,が必要なことに注意すること
    ```
    self.set_signs(
        [
            Sign(x=1.0, y=0.0, name="original"),
            Sign(x=2.0, y=0.0, name="original"),
            Sign(x=3.0, y=1.0, name="warn")
        ]
    )
    ```



In [ ]:
class Mission3Base(MissionBase):
    def __init__(self):
        super().__init__(type_b_circuit, t_max=80)
        self.goals = [
            GoalCircle((2, 0.0), 0.2, should_stop=False),
            GoalCircle((4.3, 1.0), 0.2, should_stop=False),
            GoalCircle((3.2, 2.2), 0.2, should_stop=False),
            GoalCircle((1.7, 1.0), 0.2, should_stop=False),
            GoalCircle((0.0, 0.8), 0.2, should_stop=False),
            GoalCircle((0.0, 0.0), 0.2, should_stop=False),
        ]
        self.initial_xy = (0.0, 0.0)
        self.random_d_xy = (0.1, 0.1)
        self.random_d_yaw_deg = 5

print("最大速度", prop.max_velocity, "m/秒")
print("最大回転速度", prop.max_rotate_deg, "度/秒")
MissionDrawer(Mission3Base()).show()


In [ ]:
class Mission3(Mission3Base):
    def __init__(self):
        super().__init__()
        self.set_signs(
            [
                ### 標識を変更するにはここから下を書き換える
                Sign(x=1.1, y=-0.1, name="original"),
                ### 標識を変更するにはここより上を書き換える
            ]
        )

    @staticmethod
    def command_func(alive, *, move, rotate, search, **kwargs):
        ######## ここから下にプログラムを書こう
        while alive():
            pos = search()
            if pos is None:
                move(v=0)
            else:
                if pos.name == "original":
                    move(v=0.2)
        ######## ここより上にプログラムを書こう


sim = CarSim(prop, Mission3())
sim.run()
SimDrawer(sim).show()

# ヒント

- 標識の方向に進むには、車を標識の方を向くように回転させることが必要
- 車を標識の方を向くようにするには・・・
  - もし標識が車より右にあるなら右に回転
  - もし標識が車より左にあるなら左に回転
  - もし標識が車のほぼ正面にあるなら、まっすぐ進む
- 上記を繰り返せば、標識に向かって進んでいくが・・・
- ある標識のところまで進んだあとに、次の標識が視界内に入っていないとそれ以上進めなくなってしまう
- ある標識のところまで進んだあとに、標識が見つからない場合は、回転して標識をみつけよう
  - 左にカーブする位置にはsign1を置き、右カーブする位置にはsign2を置いた上で、
  - sign1を最後にみた後であれば左回転、sign2を最後に見た後であれば右回転という条件分岐をしよう
  - それで「標識が見つかったら、標識に向かって進む」を繰り返せば一周できる
